# P1-17c · Paper 1 v2: converged retraining (E1) and bottleneck-dependent variants (E2) — session A
**Runtime: T4 GPU** (or L4). Run all. Two copies (sessions A and B) run in parallel and share the work through lock files.
Resumable: after a disconnect or an out-of-memory error, disconnect and delete the runtime, reconnect and **Run all**.
Run notebooks P1-16b (DMID) and P1-16c (mini-MIAS) first. QSEG_FINAL is read only; outputs (numbers only) go to `MyDrive/P1_v2/runs/`.

```
Paper 1 · v2 experiments (convergence and bottleneck dependence), written before any of these runs was trained.

Motivation (from the first analysis): hybrid runs were trained for at most 50 epochs with a learning-rate schedule on the
validation loss, several stopped very early, and training curves were still improving; the bottleneck code was hardly used
because the skip connections suffice for the rule-based target.

E1 (convergence): HQB-UNet, HQB-cls, HQB-free with skip connections (Paper 1 architectures, unchanged), seeds 0-4.
E2 (bottleneck dependence): the same three models WITHOUT skip connections (all information passes the four-value code),
    seeds 0-2.
Training (identical for every run): Adam 1e-4, batch 16, BCE + soft Dice, fp32, Paper 1 augmentation and seed handling,
    up to 200 epochs; ReduceLROnPlateau on the TRAINING loss (factor 0.5, patience 4, min 1e-7); early stopping on the
    validation loss (patience 10, active from epoch 15) with the best validation weights restored and tested.
Evaluation of every run: 84 test images (20 patients); external DMID set (rule-based masks, no fine-tuning);
    external mini-MIAS set scored against EXPERT-corrected breast + pectoral masks (Zenodo 10149914), no fine-tuning:
    per-image Dice; severing of the four-value code (mean training code; codes of random training images, assignments
    A1-A5 = numpy default_rng(0..4)); for HQB-UNet, finite-shot readout (100, 1,000 and 10,000 shots, 5 draws each).
Primary estimands: mean per-image test Dice of HQB-UNet - HQB-cls and HQB-UNet - HQB-free, separately in E1 and E2,
    crossed hierarchical bootstrap over patients and seeds (10,000 replicates).
Secondary: contrasts on mini-MIAS against expert masks and on DMID; severing drops (E2 serves as a positive control for the severing test);
    shot-noise sensitivity.
All runs are reported; none is discarded. Weights are kept only on the VM and deleted after evaluation;
    test, DMID and mini-MIAS probability maps (uint8) are kept for a later comparison with further reference annotations.
```


In [ ]:
# 1 · Settings, Drive, protocol record
SESSION = 'A'
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, time, shutil, random, gc
import numpy as np, pandas as pd
ROOT  = '/content/drive/MyDrive/QSEG_FINAL'        # read only
OUT   = '/content/drive/MyDrive/P1_v2'
EXT   = f'{OUT}/dmid_ext'
LOCAL = '/content/qseg_local'
TMP   = '/content/p1v2_tmp'
assert not OUT.startswith(ROOT)
os.makedirs(OUT + '/runs', exist_ok=True); os.makedirs(TMP, exist_ok=True)
assert os.path.exists(EXT + '/manifest.csv') and os.path.exists(OUT + '/mias_ext/manifest.csv'), 'Run notebooks P1-16b and P1-16c first'
PROTO = '''Paper 1 · v2 experiments (convergence and bottleneck dependence), written before any of these runs was trained.

Motivation (from the first analysis): hybrid runs were trained for at most 50 epochs with a learning-rate schedule on the
validation loss, several stopped very early, and training curves were still improving; the bottleneck code was hardly used
because the skip connections suffice for the rule-based target.

E1 (convergence): HQB-UNet, HQB-cls, HQB-free with skip connections (Paper 1 architectures, unchanged), seeds 0-4.
E2 (bottleneck dependence): the same three models WITHOUT skip connections (all information passes the four-value code),
    seeds 0-2.
Training (identical for every run): Adam 1e-4, batch 16, BCE + soft Dice, fp32, Paper 1 augmentation and seed handling,
    up to 200 epochs; ReduceLROnPlateau on the TRAINING loss (factor 0.5, patience 4, min 1e-7); early stopping on the
    validation loss (patience 10, active from epoch 15) with the best validation weights restored and tested.
Evaluation of every run: 84 test images (20 patients); external DMID set (rule-based masks, no fine-tuning);
    external mini-MIAS set scored against EXPERT-corrected breast + pectoral masks (Zenodo 10149914), no fine-tuning:
    per-image Dice; severing of the four-value code (mean training code; codes of random training images, assignments
    A1-A5 = numpy default_rng(0..4)); for HQB-UNet, finite-shot readout (100, 1,000 and 10,000 shots, 5 draws each).
Primary estimands: mean per-image test Dice of HQB-UNet - HQB-cls and HQB-UNet - HQB-free, separately in E1 and E2,
    crossed hierarchical bootstrap over patients and seeds (10,000 replicates).
Secondary: contrasts on mini-MIAS against expert masks and on DMID; severing drops (E2 serves as a positive control for the severing test);
    shot-noise sensitivity.
All runs are reported; none is discarded. Weights are kept only on the VM and deleted after evaluation;
    test, DMID and mini-MIAS probability maps (uint8) are kept for a later comparison with further reference annotations.'''
proto = OUT + '/PROTOCOL_prespecified.txt'
if not os.path.exists(proto):
    open(proto, 'w').write(PROTO + '\nwritten (UTC): ' + time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime()) + '\n')
print(open(proto).read().splitlines()[-1])


In [ ]:
# 2 · Data (local copy of the processed Paper 1 data; external DMID set)
if not os.path.exists(LOCAL + '/data/manifest.csv'):
    shutil.copytree(ROOT + '/results/data', LOCAL + '/data', ignore=shutil.ignore_patterns('*.npy', '*.h5'))
sys.path.insert(0, ROOT + '/code')
import tensorflow as tf
from tensorflow.keras import layers, models
assert tf.config.list_physical_devices('GPU'), 'No GPU: Runtime -> Change runtime type -> T4 (or L4) GPU'
from qseg.config import Config
from qseg.models import conv_block, output_layer, TFCircuitLayer
from qseg.datasets import load_manifest, make_dataset, load_arrays
from qseg.evaluation import per_image_overlap
from qseg.train import bce_dice_loss, soft_dice
cfg = Config(work_dir=LOCAL)
man = load_manifest(cfg)
X = {s: load_arrays(cfg, s, man) for s in ('train', 'val', 'test')}
em = pd.read_csv(EXT + '/manifest.csv'); em = em[em.duplicate_of.fillna('') == ''].reset_index(drop=True)
def _png(p, mask):
    a = tf.image.decode_png(tf.io.read_file(p), channels=1).numpy()
    return (a > 127).astype(np.float32) if mask else a.astype(np.float32) / 255.0
XE = np.stack([_png(f'{EXT}/images/{f}.png', False) for f in em.file_id])
YE = np.stack([_png(f'{EXT}/masks/{f}.png', True) for f in em.file_id])
MIAS = f'{OUT}/mias_ext'
mm = pd.read_csv(MIAS + '/manifest.csv'); mm = mm[mm.status == 'ok'].reset_index(drop=True)
XM = np.stack([_png(f'{MIAS}/images/{f}.png', False) for f in mm.file_id])
YM = {k: np.stack([_png(f'{MIAS}/{sub}/{f}.png', True) for f in mm.file_id])
      for k, sub in (('expert', 'masks_expert'), ('expert_breast', 'masks_expert_breast'), ('proxy', 'masks_proxy'))}
print({s: len(v[0]) for s, v in X.items()}, '| external DMID', XE.shape, '| mini-MIAS', XM.shape, '| TF', tf.__version__)


In [ ]:
# 3 · Models (Paper 1 hybrid architecture; optional removal of the skip connections) and evaluation helpers
class CodeSwitch(layers.Layer):
    """Identity during training; at test time can replace the four-value code by supplied codes."""
    def build(self, input_shape):
        self.on = self.add_weight(shape=(), initializer='zeros', trainable=False, name='on')
        self.buf = self.add_weight(shape=(cfg.batch_size, 4), initializer='zeros', trainable=False, name='buf')
    def call(self, z):
        n = tf.shape(z)[0]
        return z * (1.0 - self.on) + self.buf[:n] * self.on

def build(kind, skips=True, filters=(32, 64, 128, 256, 512), dropouts=(0.1, 0.1, 0.2, 0.2, 0.3)):
    inp = layers.Input((256, 256, 1), name='input_image')
    sk, x = [], inp
    for i, (f, d) in enumerate(zip(filters[:-1], dropouts[:-1])):
        x = conv_block(x, f, d, name=f'enc{i + 1}'); sk.append(x)
        x = layers.MaxPooling2D(2, name=f'pool{i + 1}')(x)
    x = conv_block(x, filters[-1], dropouts[-1], name='bottleneck')
    z = layers.Dense(4, activation='tanh', name='pre_dense')(layers.GlobalAveragePooling2D(name='gap')(x))
    if kind == 'quantum':   z = TFCircuitLayer(4, 2, name='quantum_circuit')(z)
    elif kind == 'classical': z = layers.Dense(4, activation='tanh', name='classical_replacement')(z)
    z = CodeSwitch(name='code_switch')(z)
    q = layers.Dense(512, activation='relu', name='post_dense')(z)
    x = layers.Reshape((16, 16, filters[-1]), name='q_reshape')(
        layers.Dense(16 * 16 * filters[-1], activation='relu', name='q_expand')(q))
    for i, (f, s, d) in enumerate(zip(reversed(filters[:-1]), reversed(sk), reversed(dropouts[:-1]))):
        x = layers.UpSampling2D(2, name=f'up{i + 1}')(x)
        if skips: x = layers.Concatenate(name=f'concat{i + 1}')([x, s])
        x = conv_block(x, f, d, name=f'dec{i + 1}')
    return models.Model(inp, output_layer(x), name=f'HQB_{kind}_{"skip" if skips else "noskip"}')

def set_seeds(seed):
    os.environ['PYTHONHASHSEED'] = str(seed); random.seed(seed); np.random.seed(seed); tf.keras.utils.set_random_seed(seed)

def predict(model, x, codes=None):
    sw = model.get_layer('code_switch'); out = []
    for a in range(0, len(x), cfg.batch_size):
        xb = x[a:a + cfg.batch_size]
        if codes is None: sw.on.assign(0.0)
        else:
            buf = np.zeros((cfg.batch_size, 4), np.float32); buf[:len(xb)] = codes[a:a + len(xb)]
            sw.buf.assign(buf); sw.on.assign(1.0)
        out.append(model(xb, training=False).numpy()[..., 0])
    sw.on.assign(0.0)
    return np.concatenate(out)

def code_model(model):
    return tf.keras.Model(model.input, model.get_layer('code_switch').input)

def dice(Y, p):
    return per_image_overlap((Y[..., 0] > 0.5).astype(np.uint8), (p >= cfg.threshold).astype(np.uint8))[:, 0]

def shots(z, n, rng):
    """Finite-shot estimate of <Z> per qubit: p(+1) = (1 + z)/2, n shots."""
    p = np.clip((1 + z) / 2, 0, 1)
    return 2 * rng.binomial(n, p) / n - 1

def evaluate(model, kind, x, Y, z_tr):
    cm = code_model(model)
    z = cm.predict(x, batch_size=cfg.batch_size, verbose=0)
    d0 = dice(Y, predict(model, x))
    r = pd.DataFrame({'dice': d0})
    r['dice_check_switch'] = dice(Y, predict(model, x, z))             # must equal dice (sanity check)
    r['drop_const'] = d0 - dice(Y, predict(model, x, np.repeat(z_tr.mean(0)[None], len(x), 0)))
    for k in range(5):
        idx = np.random.default_rng(k).integers(0, len(z_tr), len(x))
        r[f'drop_swap{k}'] = d0 - dice(Y, predict(model, x, z_tr[idx]))
    r['drop_swapavg'] = r[[f'drop_swap{k}' for k in range(5)]].mean(1)
    if kind == 'quantum':
        for n in (100, 1000, 10000):
            ds = [dice(Y, predict(model, x, shots(z, n, np.random.default_rng(100 + k)))) for k in range(5)]
            r[f'dice_shots{n}'] = np.mean(ds, 0)
    r['code_sd'] = float(z.std(0).mean())
    return r


In [ ]:
# 4 · Training and evaluation of all runs (resumable; two sessions share the work through lock files)
KIND = {'hq': 'quantum', 'hc': 'classical', 'hf': 'none'}
RUNS = [(f'E1_{m}', KIND[m], True, s) for s in range(5) for m in ('hq', 'hc', 'hf')] + \
       [(f'E2_{m}', KIND[m], False, s) for s in range(3) for m in ('hq', 'hc', 'hf')]
LOCK_H = 4
def claim(d):
    if os.path.exists(d + '/run.json'): return False
    lk = d + '/.lock'
    if os.path.exists(lk):
        who, t = open(lk).read().split()
        if who != SESSION and time.time() - float(t) < LOCK_H * 3600: return False
    os.makedirs(d, exist_ok=True); open(lk, 'w').write(f'{SESSION} {time.time()}')
    time.sleep(20)
    return open(lk).read().split()[0] == SESSION

def run(name, kind, skips, seed):
    d = f'{OUT}/runs/{name}_seed{seed}'
    if not claim(d): return
    tf.keras.backend.clear_session(); gc.collect()
    tf.keras.mixed_precision.set_global_policy('float32'); set_seeds(seed)
    tr, _ = make_dataset(cfg, 'train', seed, man); va, _ = make_dataset(cfg, 'val', seed, man)
    model = build(kind, skips)
    model.compile(optimizer=tf.keras.optimizers.Adam(cfg.learning_rate), loss=bce_dice_loss, metrics=[soft_dice], jit_compile=False)
    ck = f'{TMP}/{name}_{seed}.weights.h5'
    cbs = [tf.keras.callbacks.TerminateOnNaN(),
           tf.keras.callbacks.ModelCheckpoint(ck, monitor='val_loss', save_best_only=True, save_weights_only=True),
           tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, start_from_epoch=15),
           tf.keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5, patience=4, min_lr=1e-7),
           tf.keras.callbacks.CSVLogger(d + '/history.csv')]
    t0 = time.time()
    h = model.fit(tr, validation_data=va, epochs=200, callbacks=cbs, verbose=0)
    secs = time.time() - t0
    model.load_weights(ck)
    z_tr = code_model(model).predict(X['train'][0], batch_size=cfg.batch_size, verbose=0)
    xt, yt, dft = X['test']
    te = evaluate(model, kind, xt, yt, z_tr); te.insert(0, 'file_id', dft.file_id.values); te.insert(1, 'patient_id', dft.patient_id.values)
    ex = evaluate(model, kind, XE, YE, z_tr); ex.insert(0, 'file_id', em.file_id.values)
    pm = predict(model, XM)
    mi = pd.DataFrame({'file_id': mm.file_id.values, 'dice_expert': dice(YM['expert'], pm),
                       'dice_expert_breast': dice(YM['expert_breast'], pm), 'dice_proxy': dice(YM['proxy'], pm)})
    mi.to_csv(d + '/mias_per_image.csv', index=False)
    np.savez_compressed(d + '/test_prob_uint8.npz', prob=(predict(model, xt) * 255).round().astype(np.uint8), file_id=dft.file_id.values)
    np.savez_compressed(d + '/dmid_prob_uint8.npz', prob=(predict(model, XE) * 255).round().astype(np.uint8), file_id=em.file_id.values)
    np.savez_compressed(d + '/mias_prob_uint8.npz', prob=(pm * 255).round().astype(np.uint8), file_id=mm.file_id.values)
    va_d = dice(X['val'][1], predict(model, X['val'][0]))
    te.to_csv(d + '/test_per_image.csv', index=False); ex.to_csv(d + '/dmid_per_image.csv', index=False)
    hist = pd.DataFrame(h.history); vl = hist['val_loss'].to_numpy()
    lr = hist['learning_rate'] if 'learning_rate' in hist else hist.get('lr', pd.Series([np.nan]))
    r = dict(name=name, kind=kind, skips=skips, seed=seed, epochs_run=len(vl), best_epoch=int(np.nanargmin(vl)) + 1,
             final_lr=float(lr.iloc[-1]), minutes=round(secs / 60, 1), params=int(model.count_params()),
             val_dice=float(va_d.mean()), test_dice=float(te.dice.mean()), dmid_dice=float(ex.dice.mean()),
             mias_dice_expert=float(mi.dice_expert.mean()), mias_dice_proxy=float(mi.dice_proxy.mean()),
             test_drop_const=float(te.drop_const.mean()), test_drop_swapavg=float(te.drop_swapavg.mean()),
             switch_check_max=float(np.abs(te.dice - te.dice_check_switch).max()), session=SESSION)
    json.dump(r, open(d + '/run.json', 'w'), indent=2)
    os.remove(ck); del model; tf.keras.backend.clear_session(); gc.collect()
    done = sum(os.path.exists(f'{OUT}/runs/{a}_seed{s}/run.json') for a, _, _, s in RUNS)
    print(time.strftime('%H:%M UTC', time.gmtime()), f"| {name} seed {seed}: {r['epochs_run']} ep (best {r['best_epoch']}), "
          f"{r['minutes']} min | test Dice {r['test_dice']:.4f} | DMID {r['dmid_dice']:.4f} | MIAS expert {r['mias_dice_expert']:.4f} | drop swap {r['test_drop_swapavg']:.4f} "
          f"| finished {done} / {len(RUNS)}", flush=True)

order = RUNS if SESSION == 'A' else RUNS[::-1]
for args in order:
    run(*args)
left = [a for a in RUNS if not os.path.exists(f'{OUT}/runs/{a[0]}_seed{a[3]}/run.json')]
print(f'nothing left for this session; {len(left)} run(s) still running in the other session')


In [ ]:
# 5 · Overview of finished runs
rows = [json.load(open(f'{OUT}/runs/{a}_seed{s}/run.json')) for a, _, _, s in RUNS if os.path.exists(f'{OUT}/runs/{a}_seed{s}/run.json')]
ov = pd.DataFrame(rows); ov.to_csv(OUT + '/overview.csv', index=False)
print(ov.groupby('name')[['test_dice', 'dmid_dice', 'mias_dice_expert', 'test_drop_const', 'test_drop_swapavg', 'epochs_run', 'best_epoch']].mean().round(4).to_string())
print(ov.groupby('name').size().to_string())


In [ ]:
# 6 · Auto-shutdown
from google.colab import drive, runtime
drive.flush_and_unmount(); print('Drive flushed'); time.sleep(30); runtime.unassign()
